# CS 1090B Project — Data Loading & Preprocessing

*Group project number:* 21

*Group members:* You Wu, Jenny Yang, Xuyang Sun, Yifan Xu, Jiaxi Liu

---

This notebook covers the **complete data pipeline** from raw dataset to the two
preprocessed artifacts consumed by the main modelling notebook:

| Output | Path on Drive | Used by |
|--------|--------------|--------|
| `ds_cleaned` | `.../cs109b_project/ds_cleaned` | EDA section of final notebook |
| `ds_cleaned_final` | `.../cs109b_project/ds_cleaned_final` | Model training section of final notebook |

**Pipeline overview**

```
HuggingFace (ykumards/open-i)
        │
        ▼  Step 1 – Image preprocessing
        │  • Resize every image to 224×224 RGB
        │  • Drop samples where either view cannot be decoded
        │  • Result: 3 388 valid samples  (463 dropped)
        │
        ▼  Step 2 – Text preprocessing
        │  • Remove leading numbering ("1.", "1)")
        │  • Replace subsequent numbered items with [SEP]
        │  • Produces: findings_refined, impression_refined
        │
        ▼  Save ──► ds_cleaned
        │
        ▼  Step 3 – Feature engineering
        │  • Strip XXXX de-identification tokens
        │  • Normalise [SEP] → ". "
        │  • Produces: impression_final  (target label for models)
        │
        ▼  Save ──► ds_cleaned_final
```

# 1. Setup

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings('ignore')

import os, io, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import torch
from PIL import Image
from wordcloud import WordCloud
from datasets import load_dataset, load_from_disk
import PIL

# Reproducibility
random.seed(109)
np.random.seed(109)
torch.manual_seed(109)

In [ ]:
# Mount Google Drive (run in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

NOTEBOOK_ROOT = Path("/content/drive/MyDrive/cs109b_project")
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive mounted. Project root:", NOTEBOOK_ROOT)

# 2. Data Description

The original dataset is the **Open-I Indiana University Chest X-ray** collection
loaded from Hugging Face (`ykumards/open-i`). It is a multimodal dataset that
pairs medical images with radiology report text and clinical metadata.

At the study level the dataset contains **3 999 unique study IDs** (`uid`).
At the image level there are **7 466 unique image files**, split approximately
50/50 between frontal and lateral projections.

Key columns:

| Column | Description |
|--------|-------------|
| `uid` | Unique study identifier |
| `MeSH` | Standardised medical subject headings |
| `Problems` | Identified medical conditions |
| `image` | Imaging exam description |
| `indication` | Clinical reason for the X-ray |
| `comparison` | Reference to prior studies |
| `findings` | Detailed radiology observations |
| `impression` | Short diagnostic conclusion |
| `img_frontal` | Frontal X-ray as raw bytes |
| `img_lateral` | Lateral X-ray as raw bytes |

# 3. Load Raw Dataset from Hugging Face

In [ ]:
ds = load_dataset("ykumards/open-i")
print(ds)
ds['train'].to_pandas().head()

# 4. Data Exploration

## 4.1 Visualise Sample Images

Image columns are stored as raw byte arrays. We decode them with `PIL.Image`
and `io.BytesIO` and display the first three frontal/lateral pairs.

In [ ]:
def load_image_from_byte_array(byte_array):
    return Image.open(io.BytesIO(byte_array))

fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for i in range(3):
    img_frontal = load_image_from_byte_array(ds['train'][i]['img_frontal'])
    img_lateral = load_image_from_byte_array(ds['train'][i]['img_lateral'])

    axes[i, 0].imshow(img_frontal, cmap='gray')
    axes[i, 0].set_title(f'Sample {i+1} – Frontal')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img_lateral, cmap='gray')
    axes[i, 1].set_title(f'Sample {i+1} – Lateral')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 4.2 Image Availability Check

We verify that `img_frontal` and `img_lateral` columns are present and count
how many samples have both, only one, or neither view.

In [ ]:
df = ds['train'].to_pandas().copy()

def has_image(byte_array):
    return byte_array is not None and len(byte_array) > 0

df['has_frontal'] = df['img_frontal'].apply(has_image)
df['has_lateral'] = df['img_lateral'].apply(has_image)
df['num_images']  = df['has_frontal'].astype(int) + df['has_lateral'].astype(int)

def image_case(row):
    if row['has_frontal'] and row['has_lateral']:    return 'both_images'
    if row['has_frontal'] and not row['has_lateral']: return 'frontal_only'
    if not row['has_frontal'] and row['has_lateral']: return 'lateral_only'
    return 'no_image'

df['image_case'] = df.apply(image_case, axis=1)

summary = (df['image_case'].value_counts(dropna=False)
             .rename_axis('image_case').reset_index(name='count'))
summary['percent'] = (summary['count'] / len(df) * 100).round(2)
display(summary)

The majority of samples (~88%) have both frontal and lateral images. A smaller portion has only one view, and these are handled in the filtering step below.

## 4.3 Missing Value Check

In [ ]:
key_cols = ['img_frontal', 'img_lateral', 'findings', 'impression', 'MeSH', 'Problems']

def is_missing(x):
    if x is None: return True
    if isinstance(x, str):  return x.strip() == ''
    if isinstance(x, list): return len(x) == 0
    return False

print('=== Missing Value Summary ===')
for col in key_cols:
    values = df[col]
    n = sum(is_missing(x) for x in values)
    print(f'  {col:20s}: {n:4d}  ({n/len(values):.2%})')

`findings` has the highest missingness (13.35%), while `impression` is nearly complete (0.80%). This motivates using `impression` as the generation target. `img_lateral` is missing for ~7.8% of samples; these are removed in the next step.

# 5. Image Preprocessing

We resize every image to **224 × 224 RGB** and filter out samples where
either view cannot be decoded.

In [ ]:
def preprocess_image(byte_array, size=(224, 224)):
    """Decode bytes → PIL Image, convert to RGB, resize."""
    img = Image.open(io.BytesIO(byte_array)).convert('RGB')
    return img.resize(size, Image.Resampling.LANCZOS)

# Quick sanity-check: display original vs resized
sample_bytes = ds['train'][0]['img_frontal']
resized_img  = preprocess_image(sample_bytes)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
orig = Image.open(io.BytesIO(sample_bytes))
ax[0].imshow(orig, cmap='gray');  ax[0].set_title(f'Original  {orig.size}');  ax[0].axis('off')
ax[1].imshow(resized_img);        ax[1].set_title(f'Resized  {resized_img.size}'); ax[1].axis('off')
plt.show()

## 5.1 Apply to Full Dataset and Filter Invalid Samples

In [ ]:
def transform_images(example):
    try:
        example['img_frontal_processed'] = preprocess_image(example['img_frontal'])
        example['img_lateral_processed'] = preprocess_image(example['img_lateral'])
        example['is_valid'] = True
    except (PIL.UnidentifiedImageError, ValueError, TypeError):
        example['img_frontal_processed'] = None
        example['img_lateral_processed'] = None
        example['is_valid'] = False
    return example

ds_processed = ds['train'].map(transform_images)

skipped = ds_processed.filter(lambda x: not x['is_valid'])
print(f'Skipped samples : {len(skipped)}')
print(f'Skipped UIDs    : {skipped["uid"]}')

ds_cleaned = ds_processed.filter(lambda x: x['is_valid'])
print(f'\nOriginal size : {len(ds["train"])}')
print(f'Cleaned size  : {len(ds_cleaned)}')

463 samples are removed because at least one required image cannot be decoded.
The cleaned dataset retains **3 388 valid studies**, each with a 224 × 224 frontal
and lateral image.

# 6. Text Preprocessing

Before modelling we clean the `findings` and `impression` columns and produce
two new columns: `findings_refined` and `impression_refined`.

## 6.1 Text Quality Check

In [ ]:
def is_empty_text(text):   return text is None or str(text) == ''
def is_whitespace_only(t): return t is not None and str(t) != '' and str(t).strip() == ''
def word_count(text):
    if text is None: return 0
    return len(str(text).strip().split())
def has_suspicious_chars(text):
    if text is None: return False
    return bool(re.search(r'[^\x00-\x7F]|\\x[0-9a-fA-F]{2}', str(text)))

def text_quality_summary(dataset, col_name, short_threshold=3, long_threshold=80):
    texts = dataset[col_name]
    lengths = [word_count(x) for x in texts if x is not None and str(x).strip()]
    return pd.DataFrame([{
        'column'              : col_name,
        'total_samples'       : len(texts),
        'empty_count'         : sum(is_empty_text(x) for x in texts),
        'whitespace_only'     : sum(is_whitespace_only(x) for x in texts),
        'short_count'         : sum(1 for x in texts if 0 < word_count(x) < short_threshold),
        'long_count'          : sum(1 for x in texts if word_count(x) > long_threshold),
        'suspicious_chars'    : sum(has_suspicious_chars(x) for x in texts),
        'mean_length'         : round(sum(lengths)/len(lengths), 2) if lengths else 0,
        'max_length'          : max(lengths) if lengths else 0,
        'min_length_nonempty' : min(lengths) if lengths else 0,
    }])

summary = pd.concat([
    text_quality_summary(ds_cleaned, 'findings'),
    text_quality_summary(ds_cleaned, 'impression'),
], ignore_index=True)
display(summary)

`findings` has 445 missing entries (13%) versus only 18 for `impression` (0.5%). `impression` is also shorter and more standardised, making it the preferred generation target.

## 6.2 Structured Report Cleaning

Some impressions use a numbered list format (e.g. `1. Cardiomegaly. 2. Pleural effusion.`).
We strip the leading `1.` / `1)` marker and replace subsequent numeric markers with
a `[SEP]` separator, preserving the clinical content.

In [ ]:
# Inspect how many impressions start with a digit
numeric_start = [
    x for x in ds_cleaned['impression']
    if x is not None and len(str(x).strip()) > 0 and str(x).strip()[0].isdigit()
]
print(f'Impressions starting with a number: {len(numeric_start)}')
for imp in numeric_start[:5]:
    print(f'  • {imp}')

In [ ]:
def clean_text_with_custom_sep(text, separator=' [SEP] '):
    """Remove leading '1.' / '1)' and replace subsequent numbered items with [SEP]."""
    if text is None:
        return ''
    text = str(text).strip()
    text = re.sub(r'^1[\.\)]\s*', '', text)
    text = re.sub(r'\s*\d+[\.\)]\s*', separator, text)
    text = re.sub(r'\.\s*\.$', '.', text)
    return re.sub(r'\s+', ' ', text).strip()

def apply_refined_cleaning(example):
    example['findings_refined']   = clean_text_with_custom_sep(example['findings'])
    example['impression_refined'] = clean_text_with_custom_sep(example['impression'])
    return example

ds_cleaned = ds_cleaned.map(apply_refined_cleaning)

# Verify on a few numbered impressions
check_df = ds_cleaned.to_pandas()
mask = check_df['impression'].str.contains(r'^1[\.\)]', na=False, regex=True)
display(check_df[mask][['impression', 'impression_refined']].head(5))

Leading numbering is removed and list items are joined with `[SEP]`, preserving all clinical content in a single-string format.

# 7. Save `ds_cleaned` to Google Drive

This is the artifact loaded by **Section 2 (EDA)** of the final notebook.

Columns added so far: `img_frontal_processed`, `img_lateral_processed`, `is_valid`, `findings_refined`, `impression_refined`.

In [ ]:
SAVE_PATH_CLEANED = str(NOTEBOOK_ROOT / 'ds_cleaned')
ds_cleaned.save_to_disk(SAVE_PATH_CLEANED)
print(f'Saved ds_cleaned  →  {SAVE_PATH_CLEANED}')
print(f'Samples : {len(ds_cleaned)}')
print(f'Columns : {ds_cleaned.column_names}')

# 8. Feature Engineering → `impression_final`

We now create the **final model target label** `impression_final` by:

1. Removing `XXXX` de-identification placeholders (found in ~36% of findings,
   meaningful fraction of impressions).
2. Replacing `[SEP]` with `". "` so the text reads as natural prose.

This section mirrors *Section 3 – Feature Engineering* of the MS3 notebook.

## 8.1 Remove XXXX Tokens

In [ ]:
def remove_xxxx(text):
    if text is None: return None
    text = re.sub(r'XXXX', '', str(text), flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

ds_cleaned = ds_cleaned.map(
    lambda x: {'impression_final': remove_xxxx(x['impression_refined'])}
)

check_df = ds_cleaned.select(range(10)).to_pandas()[
    ['uid', 'impression_refined', 'impression_final']
]
display(check_df)

## 8.2 Normalise [SEP] Tokens

In [ ]:
def normalize_sep(text, sep_replacement='. '):
    if text is None: return None
    text = re.sub(r'\[SEP\]', sep_replacement, str(text), flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s*\.\s*\.', '.', text)   # remove double periods
    text = re.sub(r'\s+,', ',', text)           # remove space before commas
    return text

ds_cleaned = ds_cleaned.map(
    lambda x: {'impression_final': normalize_sep(x['impression_final'])}
)

check_df = ds_cleaned.select(range(10)).to_pandas()[
    ['uid', 'impression_refined', 'impression_final']
]
display(check_df)

`impression_final` is now free of de-identification noise and formatted as
natural prose — the target label used in all downstream models.

# 9. Save `ds_cleaned_final` to Google Drive

This is the artifact loaded by **Section 4 & 5 (Model Training)** of the final notebook.

Additional column over `ds_cleaned`: `impression_final`.

In [ ]:
SAVE_PATH_FINAL = str(NOTEBOOK_ROOT / 'ds_cleaned_final')
ds_cleaned.save_to_disk(SAVE_PATH_FINAL)
print(f'Saved ds_cleaned_final  →  {SAVE_PATH_FINAL}')
print(f'Samples : {len(ds_cleaned)}')
print(f'Columns : {ds_cleaned.column_names}')

# 10. Summary

| Step | Action | Output |
|------|--------|--------|
| Load | `load_dataset("ykumards/open-i")` | 3 999 raw studies |
| Image preprocessing | Resize to 224×224, filter unreadable | 3 388 valid samples, `img_frontal_processed`, `img_lateral_processed`, `is_valid` |
| Text preprocessing | Strip leading numbers, add `[SEP]` | `findings_refined`, `impression_refined` |
| **Save** | `ds_cleaned` | Used by EDA section of final notebook |
| Feature engineering | Remove `XXXX`, normalise `[SEP]` → `". "` | `impression_final` |
| **Save** | `ds_cleaned_final` | Used by model training section of final notebook |

Both saved artefacts are located under:
```
/content/drive/MyDrive/cs109b_project/
├── ds_cleaned/          ← loaded by: load_from_disk(NOTEBOOK_ROOT / 'ds_cleaned')
└── ds_cleaned_final/    ← loaded by: load_from_disk(NOTEBOOK_ROOT / 'ds_cleaned_final')
```